# Practice Lab: Math Foundations 4 - Transformer Plumbing

8 exercises on LayerNorm, RMSNorm, positional encoding, residuals.

In [ ]:
!pip install -q numpy torch matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

## Exercise 1 - LayerNorm by hand

**Task:** Normalize `x` by hand (subtract the mean, divide by `sqrt(var + eps)` using the biased variance), then confirm it matches `nn.LayerNorm(4)`.

**Expected:** both lines print the same values, mean 0 and unit scale.

In [ ]:
# ✏️ YOUR CODE HERE
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
# TODO: mean = mean of x over the last dim (keepdim=True)
# TODO: var  = variance of x over the last dim (keepdim=True, unbiased=False)
# TODO: my_ln = (x - mean) / sqrt(var + 1e-5); then print it rounded to 3 dp

# TODO: build ln = nn.LayerNorm(4) and print ln(x) rounded to 3 dp to compare

<details><summary>✅ Solution</summary>



In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
mean = x.mean(dim=-1, keepdim=True)
var = x.var(dim=-1, keepdim=True, unbiased=False)
my_ln = (x - mean) / torch.sqrt(var + 1e-5)
print(f'by hand: {my_ln.round(decimals=3)}')

ln = nn.LayerNorm(4)
print(f'pytorch: {ln(x).round(decimals=3)}')

</details>

---

## Exercise 2 - RMSNorm vs LayerNorm

**Task:** Apply both `nn.LayerNorm(8)` and `nn.RMSNorm(8)` to `x`. Print the LayerNorm output mean and std, and the RMSNorm output mean and RMS.

**Expected:** LayerNorm mean ~0; RMSNorm mean is NOT recentered (only rescaled), RMS ~1.

In [ ]:
# ✏️ YOUR CODE HERE
x = torch.tensor([[3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0]])
# TODO: ln = nn.LayerNorm(8); rms = nn.RMSNorm(8); apply both to x
# TODO: print LayerNorm mean and std (unbiased=False)
# TODO: print RMSNorm mean and RMS = sqrt(mean(out**2))

<details><summary>✅ Solution</summary>



In [ ]:
x = torch.tensor([[3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0]])
ln = nn.LayerNorm(8)
rms = nn.RMSNorm(8)
ln_out = ln(x); rms_out = rms(x)
print(f'LayerNorm: mean={ln_out.mean().item():.3f}, std={ln_out.std(unbiased=False).item():.3f}')
print(f'RMSNorm:   mean={rms_out.mean().item():.3f}, RMS={(rms_out**2).mean().sqrt().item():.3f}')

</details>

---

## Exercise 3 - Sinusoidal PE heatmap

**Task:** Implement `sinusoidal_pe(seq_len, d_model)` returning a `(seq_len, d_model)` tensor with sin on even dims and cos on odd dims, then plot it as a heatmap and print its shape.

**Expected:** a smooth striped heatmap; `PE shape: torch.Size([16, 32])`.

In [ ]:
# ✏️ YOUR CODE HERE
def sinusoidal_pe(seq_len, d_model):
    # TODO: pos = arange(seq_len).unsqueeze(1)
    # TODO: div = exp(arange(0, d_model, 2) * (-log(10000)/d_model))
    # TODO: pe[:, 0::2] = sin(pos*div); pe[:, 1::2] = cos(pos*div); return pe
    pass

# TODO: pe = sinusoidal_pe(16, 32); plt.imshow(pe, aspect='auto', cmap='RdBu'); show; print pe.shape

<details><summary>✅ Solution</summary>



In [ ]:
def sinusoidal_pe(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    pos = torch.arange(0, seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = sinusoidal_pe(16, 32)
plt.imshow(pe.numpy(), aspect='auto', cmap='RdBu')
plt.xlabel('dimension'); plt.ylabel('position'); plt.colorbar(); plt.title('Sinusoidal PE')
plt.show()
print(f'PE shape: {pe.shape}')

</details>

---

## Exercise 4 - Vanishing gradient demo

**Task:** Complete `Block.forward` (add the residual when `use_residual`) and `grad_norm` (stack `depth` blocks, backprop from the output sum, return the input gradient norm). Compare 20 layers with and without residuals.

**Expected:** the no-residual grad norm is many orders of magnitude smaller.

In [ ]:
# ✏️ YOUR CODE HERE
class Block(nn.Module):
    def __init__(self, d, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        self.layer = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        f = self.layer(x)
        # TODO: return x + f when self.use_residual else f
        pass

def grad_norm(use_residual, depth=20):
    # TODO: seed, stack `depth` Blocks in nn.Sequential
    # TODO: x = randn(1, 8, requires_grad=True); y = blocks(x).sum(); y.backward()
    # TODO: return x.grad.norm().item()
    pass

# TODO: print grad norm for no-residual vs with-residual at depth 20

<details><summary>✅ Solution</summary>



In [ ]:
class Block(nn.Module):
    def __init__(self, d, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        self.layer = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        f = self.layer(x)
        return x + f if self.use_residual else f

def grad_norm(use_residual, depth=20):
    torch.manual_seed(0)
    blocks = nn.Sequential(*[Block(8, use_residual) for _ in range(depth)])
    x = torch.randn(1, 8, requires_grad=True)
    y = blocks(x).sum()
    y.backward()
    return x.grad.norm().item()

print(f'20 layers, NO residual:   grad norm = {grad_norm(False):.3e}')
print(f'20 layers, WITH residual: grad norm = {grad_norm(True):.3e}')

</details>

---

## Exercise 5 - Pre-LN vs Post-LN convergence

**Task:** Complete `Block2.forward` for both orderings (pre-LN normalizes inside the residual, post-LN normalizes the sum) and `train` (12 blocks, AdamW lr=1e-3, MSE to a random target). Compare final losses.

**Expected:** Pre-LN trains to a lower final loss than Post-LN at this depth.

In [ ]:
# ✏️ YOUR CODE HERE
class Block2(nn.Module):
    def __init__(self, d, pre_ln=True):
        super().__init__()
        self.pre_ln = pre_ln
        self.norm1 = nn.LayerNorm(d); self.norm2 = nn.LayerNorm(d)
        self.ffn1 = nn.Linear(d, d); self.ffn2 = nn.Linear(d, d)
    def forward(self, x):
        # TODO: pre-LN -> x = x + gelu(ffn1(norm1(x))); x = x + ffn2(norm2(x))
        # TODO: post-LN -> x = norm1(x + gelu(ffn1(x))); x = norm2(x + ffn2(x))
        pass

def train(pre_ln, n_steps=200):
    # TODO: seed; model = 12 stacked Block2; AdamW lr=1e-3
    # TODO: x, y = randn(32,16); loop MSE loss, backward, step; collect losses
    pass

# TODO: run both and print Pre-LN and Post-LN final loss

<details><summary>✅ Solution</summary>



In [ ]:
class Block2(nn.Module):
    def __init__(self, d, pre_ln=True):
        super().__init__()
        self.pre_ln = pre_ln
        self.norm1 = nn.LayerNorm(d); self.norm2 = nn.LayerNorm(d)
        self.ffn1 = nn.Linear(d, d); self.ffn2 = nn.Linear(d, d)
    def forward(self, x):
        if self.pre_ln:
            x = x + F.gelu(self.ffn1(self.norm1(x)))
            x = x + self.ffn2(self.norm2(x))
        else:
            x = self.norm1(x + F.gelu(self.ffn1(x)))
            x = self.norm2(x + self.ffn2(x))
        return x

def train(pre_ln, n_steps=200):
    torch.manual_seed(0)
    model = nn.Sequential(*[Block2(16, pre_ln) for _ in range(12)])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x = torch.randn(32, 16); y = torch.randn(32, 16)
    losses = []
    for _ in range(n_steps):
        opt.zero_grad()
        loss = F.mse_loss(model(x), y)
        loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

pre = train(True); post = train(False)
print(f'Pre-LN  final: {pre[-1]:.4f}')
print(f'Post-LN final: {post[-1]:.4f}')

</details>

---

## Exercise 6 - RoPE rotation

**Task:** Implement `rope_rotate(Q, base=10000)` that rotates each even/odd feature pair by a position-dependent angle. Confirm the per-row norm is preserved (rotation is length-preserving).

**Expected:** rotated norms equal the original norms; the assert passes.

In [ ]:
# ✏️ YOUR CODE HERE
def rope_rotate(Q, base=10000):
    # TODO: positions = arange(seq_len).unsqueeze(1)
    # TODO: freqs = 1 / (base ** (arange(0, d_head, 2)/d_head)); angles = positions*freqs
    # TODO: rotate even/odd pairs with cos/sin, reassemble, return out
    pass

torch.manual_seed(0)
Q = torch.randn(4, 8)
# TODO: Q_rot = rope_rotate(Q); print original vs rotated per-row norms
# TODO: assert torch.allclose(Q.norm(dim=-1), Q_rot.norm(dim=-1), atol=1e-5)

<details><summary>✅ Solution</summary>



In [ ]:
def rope_rotate(Q, base=10000):
    seq_len, d_head = Q.shape
    positions = torch.arange(seq_len).float().unsqueeze(1)
    freqs = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    angles = positions * freqs
    cos = torch.cos(angles); sin = torch.sin(angles)
    Q_even = Q[:, 0::2]; Q_odd = Q[:, 1::2]
    rotated_even = Q_even * cos - Q_odd * sin
    rotated_odd = Q_even * sin + Q_odd * cos
    out = torch.zeros_like(Q)
    out[:, 0::2] = rotated_even
    out[:, 1::2] = rotated_odd
    return out

torch.manual_seed(0)
Q = torch.randn(4, 8)
Q_rot = rope_rotate(Q)
print(f'original norms per row: {Q.norm(dim=-1).round(decimals=3).tolist()}')
print(f'rotated norms per row:  {Q_rot.norm(dim=-1).round(decimals=3).tolist()}')
assert torch.allclose(Q.norm(dim=-1), Q_rot.norm(dim=-1), atol=1e-5)

</details>

---

## Exercise 7 - Full Pre-LN block from scratch

**Task:** Complete `PreLNBlock.forward`: pre-norm multi-head causal attention with a residual, then pre-norm FFN with a residual. Run it on a `(2, 8, 64)` input and print the output shape.

**Expected:** `output shape: torch.Size([2, 8, 64])` (shape preserved).

In [ ]:
# ✏️ YOUR CODE HERE
class PreLNBlock(nn.Module):
    def __init__(self, d=64, n_heads=4):
        super().__init__()
        self.norm1 = nn.RMSNorm(d); self.norm2 = nn.RMSNorm(d)
        self.qkv = nn.Linear(d, 3*d)
        self.out_proj = nn.Linear(d, d)
        self.ffn = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        self.n_heads = n_heads; self.d_head = d // n_heads
    def forward(self, x):
        B, T, D = x.shape
        # TODO: h = norm1(x); q, k, v = qkv(h).chunk(3, dim=-1)
        # TODO: reshape each to (B, n_heads, T, d_head)
        # TODO: a = scaled_dot_product_attention(q, k, v, is_causal=True); merge heads
        # TODO: x = x + out_proj(a); x = x + ffn(norm2(x)); return x
        pass

torch.manual_seed(0)
block = PreLNBlock()
x = torch.randn(2, 8, 64)
# TODO: out = block(x); print output shape

<details><summary>✅ Solution</summary>



In [ ]:
class PreLNBlock(nn.Module):
    def __init__(self, d=64, n_heads=4):
        super().__init__()
        self.norm1 = nn.RMSNorm(d); self.norm2 = nn.RMSNorm(d)
        self.qkv = nn.Linear(d, 3*d)
        self.out_proj = nn.Linear(d, d)
        self.ffn = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        self.n_heads = n_heads; self.d_head = d // n_heads
    def forward(self, x):
        B, T, D = x.shape
        h = self.norm1(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        a = a.transpose(1, 2).contiguous().view(B, T, D)
        x = x + self.out_proj(a)
        x = x + self.ffn(self.norm2(x))
        return x

torch.manual_seed(0)
block = PreLNBlock()
x = torch.randn(2, 8, 64)
out = block(x)
print(f'output shape: {out.shape}')

</details>

---

## Exercise 8 - YaRN-style RoPE base scaling

**Task:** Implement `rope_rotate_scaled(Q, base=10000, scale=1.0)` (same as RoPE but with an `effective_base = base * scale`) and `attn_score_var` measuring the variance of `Q_rope @ K_rope.T`. Compare short vs long sequences, and long sequence with a larger scale.

**Expected:** longer sequences raise attention-score variance; scaling the base tempers it.

In [ ]:
# ✏️ YOUR CODE HERE
def rope_rotate_scaled(Q, base=10000, scale=1.0):
    # TODO: same rotation as Exercise 6 but freqs use effective_base = base * scale
    pass

torch.manual_seed(0)
def attn_score_var(seq_len, base=10000, scale=1.0):
    # TODO: Q, K = randn(seq_len, 16); apply rope_rotate_scaled to both
    # TODO: return (Q_rope @ K_rope.T).var().item()
    pass

# TODO: print attn var for (32, scale=1.0), (128, scale=1.0), (128, scale=4.0)

<details><summary>✅ Solution</summary>



In [ ]:
def rope_rotate_scaled(Q, base=10000, scale=1.0):
    seq_len, d_head = Q.shape
    positions = torch.arange(seq_len).float().unsqueeze(1)
    effective_base = base * scale
    freqs = 1.0 / (effective_base ** (torch.arange(0, d_head, 2).float() / d_head))
    angles = positions * freqs
    cos = torch.cos(angles); sin = torch.sin(angles)
    Q_even = Q[:, 0::2]; Q_odd = Q[:, 1::2]
    rotated_even = Q_even * cos - Q_odd * sin
    rotated_odd = Q_even * sin + Q_odd * cos
    out = torch.zeros_like(Q)
    out[:, 0::2] = rotated_even
    out[:, 1::2] = rotated_odd
    return out

torch.manual_seed(0)
def attn_score_var(seq_len, base=10000, scale=1.0):
    Q = torch.randn(seq_len, 16)
    K = torch.randn(seq_len, 16)
    Q_rope = rope_rotate_scaled(Q, base, scale)
    K_rope = rope_rotate_scaled(K, base, scale)
    return (Q_rope @ K_rope.T).var().item()

print(f'Seq 32,  scale=1.0 -> attn var: {attn_score_var(32):.2f}')
print(f'Seq 128, scale=1.0 -> attn var: {attn_score_var(128):.2f}')
print(f'Seq 128, scale=4.0 -> attn var: {attn_score_var(128, scale=4.0):.2f}')

</details>

---

## Wrap-up

Exercise 4 (vanishing gradient) is the most memorable demo. Exercise 7 (Pre-LN block) is the most useful Module 4 prep.